In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install transformers openai-clip -q

TRAIN_CSV       = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/train/train.csv" 
EVAL_CSV        = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/eval/val.csv"
TEST_CSV        = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/test_release/index_text.csv"
TRAIN_IMAGE_DIR = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/train/train_images"  
EVAL_IMAGE_DIR  = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/eval/val_no_labels/val_images"
TEST_IMAGE_DIR  = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/test_release/test_images"
OUTPUT_DIR      = "/kaggle/working/"

# CELL 2 — Imports
import os, random, zipfile
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score, classification_report
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
import clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# CELL 3 — Config & Shared Utilities

TEXT_MODEL       = "roberta-base"
CLIP_MODEL       = "ViT-B/32"

MAX_LEN          = 128
BATCH_SIZE       = 16
EPOCHS_TEXT      = 5
EPOCHS_MM        = 6
EPOCHS_COMBINED  = 3    # extra epochs when training on train+eval combined
TEXT_LR          = 2e-5
CLIP_LR          = 5e-7
HEAD_LR          = 5e-5
WEIGHT_DECAY     = 1e-2
WARMUP_RATIO     = 0.1
DROPOUT          = 0.1
CLIP_UNFREEZE_AT = 3
FUSION_DIM       = 512
CLIP_HIDDEN      = 512
TEXT_HIDDEN      = 768   # roberta-base hidden size = 768, same as bert-base

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction="none")
        pt  = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

# CELL 4 — Datasets

class TextDataset(Dataset):
    """Text-only dataset for RoBERTa.
    RoBERTa does NOT use token_type_ids — tokenizer won't return them."""
    def __init__(self, df, tokenizer, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("post_text") or "")
        enc  = self.tokenizer(
            text, max_length=MAX_LEN, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        # RoBERTa only has input_ids + attention_mask
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item


class MemeDataset(Dataset):
    """Multimodal dataset — text (RoBERTa) + image (CLIP)."""
    def __init__(self, df, tokenizer, clip_prep, image_dir, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.clip_prep = clip_prep
        self.image_dir = image_dir
        self.is_test   = is_test
        self._blank    = torch.zeros(3, 224, 224)

    def __len__(self): return len(self.df)

    def _load_img(self, fname):
        try:
            return self.clip_prep(
                Image.open(os.path.join(self.image_dir, str(fname))).convert("RGB")
            )
        except:
            return self._blank.clone()

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("post_text") or "")
        enc  = self.tokenizer(
            text, max_length=MAX_LEN, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image":          self._load_img(row["index"]),
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Models
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TextClassifier(nn.Module):
    """RoBERTa text-only classifier with CLS+mean pooling blend."""
    def __init__(self):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(TEXT_MODEL)
        self.dropout    = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(TEXT_HIDDEN, 3)

    def encode(self, input_ids, attention_mask):
        out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_f    = out.last_hidden_state[:, 0, :]
        mask     = attention_mask.unsqueeze(-1).float()
        mean_f   = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(1e-9)
        return (cls_f + mean_f) / 2

    def forward(self, input_ids, attention_mask):
        return self.classifier(self.dropout(self.encode(input_ids, attention_mask)))


class GatedMultimodalModel(nn.Module):
    """
    RoBERTa + CLIP ViT with gated cross-attention fusion.
    Gate learns per-sample: if image is a blank/useless meme,
    it weights toward text. If image is informative, it blends both.
    """
    def __init__(self):
        super().__init__()
        # Text
        self.text_enc = AutoModel.from_pretrained(TEXT_MODEL)
        # Image (CLIP visual encoder — frozen initially)
        clip_mdl, _   = clip.load(CLIP_MODEL, device="cpu")
        self.img_enc  = clip_mdl.visual.float()
        for p in self.img_enc.parameters():
            p.requires_grad = False
        # Projections to shared fusion space
        self.text_proj = nn.Sequential(
            nn.Linear(TEXT_HIDDEN, FUSION_DIM), nn.LayerNorm(FUSION_DIM),
            nn.GELU(), nn.Dropout(DROPOUT))
        self.img_proj = nn.Sequential(
            nn.Linear(CLIP_HIDDEN, FUSION_DIM), nn.LayerNorm(FUSION_DIM),
            nn.GELU(), nn.Dropout(DROPOUT))
        # Cross-attention: text queries image
        self.cross_attn = nn.MultiheadAttention(
            FUSION_DIM, num_heads=8, dropout=DROPOUT, batch_first=True)
        self.norm = nn.LayerNorm(FUSION_DIM)
        # Gate
        self.gate = nn.Sequential(
            nn.Linear(FUSION_DIM * 2, FUSION_DIM), nn.ReLU(),
            nn.Linear(FUSION_DIM, 1), nn.Sigmoid())
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_DIM, 256), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(256, 3))

    def unfreeze_clip(self):
        for p in self.img_enc.parameters():
            p.requires_grad = True
        print("  → CLIP unfrozen for fine-tuning")

    def forward(self, input_ids, attention_mask, image):
        # Text features
        out    = self.text_enc(input_ids=input_ids, attention_mask=attention_mask)
        cls_f  = out.last_hidden_state[:, 0, :]
        mask   = attention_mask.unsqueeze(-1).float()
        mean_f = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(1e-9)
        text_f = self.text_proj((cls_f + mean_f) / 2)

        # Image features
        grad_fn = torch.enable_grad if any(
            p.requires_grad for p in self.img_enc.parameters()
        ) else torch.no_grad
        with grad_fn():
            img_f = self.img_proj(self.img_enc(image).float())

        # Cross-attention: text attends to image
        q = text_f.unsqueeze(1)
        k = img_f.unsqueeze(1)
        cross, _ = self.cross_attn(q, k, k)
        cross    = self.norm(cross.squeeze(1))

        # Gating
        gate  = self.gate(torch.cat([text_f, cross], dim=-1))
        fused = gate * cross + (1 - gate) * text_f

        return self.classifier(fused)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — MODEL A: Text-Only RoBERTa
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_text_model():
    print("\n" + "="*60)
    print("STEP 1/4 — Text-Only RoBERTa (train→eval split)")
    print("="*60)

    train_df  = pd.read_csv(TRAIN_CSV)
    eval_df   = pd.read_csv(EVAL_CSV)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

    train_loader = DataLoader(TextDataset(train_df, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    eval_loader  = DataLoader(TextDataset(eval_df,  tokenizer),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model     = TextClassifier().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    scaler    = GradScaler()
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(),    "lr": TEXT_LR},
        {"params": model.classifier.parameters(), "lr": TEXT_LR * 5},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS_TEXT
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    best_f1, best_path = 0, os.path.join(OUTPUT_DIR, "best_text.pt")

    for epoch in range(1, EPOCHS_TEXT + 1):
        model.train()
        for batch in train_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(ids, mask), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

        # Eval
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for batch in eval_loader:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                logits = model(ids, mask)
                preds.extend(logits.argmax(-1).cpu().numpy())
                labs.extend(batch["label"].numpy())

        val_f1 = f1_score(labs, preds, average="macro")
        print(f"  Epoch {epoch}/{EPOCHS_TEXT}  Val F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Saved (F1: {best_f1:.4f})")

    print(classification_report(labs, preds,
          target_names=["Vaccine Critical", "Neutral", "Pro-vaccine"], digits=4))

    # Generate test logits
    test_df     = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(TextDataset(test_df, tokenizer, is_test=True),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load(best_path))
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with autocast():
                logits = model(ids, mask)
            all_logits.append(logits.cpu().float().numpy())

    text_logits = np.concatenate(all_logits, axis=0)
    np.save(os.path.join(OUTPUT_DIR, "logits_text.npy"), text_logits)
    print(f"  Text logits saved → shape {text_logits.shape}  |  Best Val F1: {best_f1:.4f}")
    return text_logits, best_f1

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — MODEL B: Multimodal RoBERTa + CLIP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_multimodal_model():
    print("\n" + "="*60)
    print("STEP 2/4 — Multimodal RoBERTa+CLIP (train→eval split)")
    print("="*60)

    train_df  = pd.read_csv(TRAIN_CSV)
    eval_df   = pd.read_csv(EVAL_CSV)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
    _, clip_prep = clip.load(CLIP_MODEL, device="cpu")

    train_loader = DataLoader(
        MemeDataset(train_df, tokenizer, clip_prep, TRAIN_IMAGE_DIR),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    eval_loader  = DataLoader(
        MemeDataset(eval_df, tokenizer, clip_prep, EVAL_IMAGE_DIR),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model     = GatedMultimodalModel().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    scaler    = GradScaler()
    optimizer = torch.optim.AdamW([
        {"params": model.text_enc.parameters(),   "lr": TEXT_LR},
        {"params": model.img_enc.parameters(),    "lr": CLIP_LR},
        {"params": model.text_proj.parameters(),  "lr": HEAD_LR},
        {"params": model.img_proj.parameters(),   "lr": HEAD_LR},
        {"params": model.cross_attn.parameters(), "lr": HEAD_LR},
        {"params": model.gate.parameters(),       "lr": HEAD_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS_MM
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    best_f1, best_path = 0, os.path.join(OUTPUT_DIR, "best_multimodal.pt")

    for epoch in range(1, EPOCHS_MM + 1):
        if epoch == CLIP_UNFREEZE_AT + 1:
            model.unfreeze_clip()

        model.train()
        for batch in train_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            img  = batch["image"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(ids, mask, img), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for batch in eval_loader:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                img  = batch["image"].to(DEVICE)
                logits = model(ids, mask, img)
                preds.extend(logits.argmax(-1).cpu().numpy())
                labs.extend(batch["label"].numpy())

        val_f1 = f1_score(labs, preds, average="macro")
        print(f"  Epoch {epoch}/{EPOCHS_MM}  Val F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Saved (F1: {best_f1:.4f})")

    print(classification_report(labs, preds,
          target_names=["Vaccine Critical", "Neutral", "Pro-vaccine"], digits=4))

    # Generate test logits
    test_df     = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(
        MemeDataset(test_df, tokenizer, clip_prep, TEST_IMAGE_DIR, is_test=True),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load(best_path))
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            img  = batch["image"].to(DEVICE)
            with autocast():
                logits = model(ids, mask, img)
            all_logits.append(logits.cpu().float().numpy())

    mm_logits = np.concatenate(all_logits, axis=0)
    np.save(os.path.join(OUTPUT_DIR, "logits_multimodal.npy"), mm_logits)
    print(f"  Multimodal logits saved → shape {mm_logits.shape}  |  Best Val F1: {best_f1:.4f}")
    return mm_logits, best_f1

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — MODEL C: Multimodal retrained on TRAIN+EVAL combined
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train_combined_model():
    """
    Retrain the multimodal model on ALL labeled data (train + eval).
    No validation possible here — we use the epoch count from best model above.
    This gives the model 12% more training data (8195 → 9219 samples).
    """
    print("\n" + "="*60)
    print("STEP 3/4 — Multimodal retrained on Train+Eval combined")
    print("  (no val — using best epoch count from Step 2)")
    print("="*60)

    train_df  = pd.read_csv(TRAIN_CSV)
    eval_df   = pd.read_csv(EVAL_CSV)
    # Combine both splits — eval images are in EVAL_IMAGE_DIR
    combined_df = pd.concat([train_df, eval_df], ignore_index=True)
    print(f"  Combined dataset size: {len(combined_df)} samples")
    print(f"  Label dist: {dict(Counter(combined_df['label'].tolist()))}")

    tokenizer    = AutoTokenizer.from_pretrained(TEXT_MODEL)
    _, clip_prep = clip.load(CLIP_MODEL, device="cpu")

    # We need images from both folders — add a split column to route correctly
    train_df["_img_dir"] = TRAIN_IMAGE_DIR
    eval_df["_img_dir"]  = EVAL_IMAGE_DIR

    class CombinedMemeDataset(Dataset):
        def __init__(self, df, tokenizer, clip_prep):
            self.df        = df.reset_index(drop=True)
            self.tokenizer = tokenizer
            self.clip_prep = clip_prep
            self._blank    = torch.zeros(3, 224, 224)

        def __len__(self): return len(self.df)

        def _load_img(self, fname, img_dir):
            try:
                return self.clip_prep(
                    Image.open(os.path.join(img_dir, str(fname))).convert("RGB"))
            except:
                return self._blank.clone()

        def __getitem__(self, idx):
            row  = self.df.iloc[idx]
            text = str(row.get("post_text") or "")
            enc  = self.tokenizer(text, max_length=MAX_LEN, padding="max_length",
                                  truncation=True, return_tensors="pt")
            return {
                "input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "image":          self._load_img(row["index"], row["_img_dir"]),
                "label":          torch.tensor(int(row["label"]), dtype=torch.long),
            }

    combined_df_with_dirs = pd.concat([train_df, eval_df], ignore_index=True)
    combined_loader = DataLoader(
        CombinedMemeDataset(combined_df_with_dirs, tokenizer, clip_prep),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    model     = GatedMultimodalModel().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    scaler    = GradScaler()
    optimizer = torch.optim.AdamW([
        {"params": model.text_enc.parameters(),   "lr": TEXT_LR},
        {"params": model.img_enc.parameters(),    "lr": CLIP_LR},
        {"params": model.text_proj.parameters(),  "lr": HEAD_LR},
        {"params": model.img_proj.parameters(),   "lr": HEAD_LR},
        {"params": model.cross_attn.parameters(), "lr": HEAD_LR},
        {"params": model.gate.parameters(),       "lr": HEAD_LR},
        {"params": model.classifier.parameters(), "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = len(combined_loader) * EPOCHS_COMBINED
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    best_path = os.path.join(OUTPUT_DIR, "best_combined.pt")

    for epoch in range(1, EPOCHS_COMBINED + 1):
        if epoch == 2:   # unfreeze CLIP at epoch 2 (shorter training)
            model.unfreeze_clip()
        model.train()
        total_loss = 0
        for batch in combined_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            img  = batch["image"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(ids, mask, img), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch}/{EPOCHS_COMBINED}  Train Loss: {total_loss/len(combined_loader):.4f}")

    torch.save(model.state_dict(), best_path)
    print(f"  Combined model saved → {best_path}")

    # Generate test logits
    test_df     = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(
        MemeDataset(test_df, tokenizer, clip_prep, TEST_IMAGE_DIR, is_test=True),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            img  = batch["image"].to(DEVICE)
            with autocast():
                logits = model(ids, mask, img)
            all_logits.append(logits.cpu().float().numpy())

    combined_logits = np.concatenate(all_logits, axis=0)
    np.save(os.path.join(OUTPUT_DIR, "logits_combined.npy"), combined_logits)
    print(f"  Combined logits saved → shape {combined_logits.shape}")
    return combined_logits

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — Final Ensemble: weight by val F1
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def make_final_submission(text_logits, text_f1, mm_logits, mm_f1, combined_logits):
    """
    Weighted ensemble — each model's weight is proportional to its val F1.
    Combined model has no val F1 so it gets the multimodal model's F1 as proxy.

    Example: text=0.81, mm=0.83 → weights = 0.81/(0.81+0.83+0.83), etc.
    """
    print("\n" + "="*60)
    print("STEP 4/4 — Weighted Ensemble → Final Submission")
    print("="*60)

    # Weights proportional to val F1
    w_text = text_f1
    w_mm   = mm_f1
    w_comb = mm_f1    # proxy: combined model ≥ mm model
    total  = w_text + w_mm + w_comb

    print(f"  Model weights (F1-proportional):")
    print(f"    Text-only RoBERTa  : {w_text/total:.3f}  (val F1={text_f1:.4f})")
    print(f"    Multimodal         : {w_mm/total:.3f}  (val F1={mm_f1:.4f})")
    print(f"    Combined (no val)  : {w_comb/total:.3f}  (proxy F1={mm_f1:.4f})")

    def softmax_np(x):
        return torch.softmax(torch.tensor(x), dim=-1).numpy()

    final_probs = (
        (w_text / total) * softmax_np(text_logits) +
        (w_mm   / total) * softmax_np(mm_logits)   +
        (w_comb / total) * softmax_np(combined_logits)
    )
    preds = final_probs.argmax(axis=1)

    test_df = pd.read_csv(TEST_CSV)
    sub_df  = pd.DataFrame({
        "index": test_df["index"].values,
        "label": preds
    }).sort_values("index").reset_index(drop=True)

    csv_path = os.path.join(OUTPUT_DIR, "predictions.csv")
    zip_path = os.path.join(OUTPUT_DIR, "submission_v2.zip")
    sub_df.to_csv(csv_path, index=False)
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(csv_path, "predictions.csv")

    print(f"\n  Label dist: {dict(pd.Series(preds).value_counts().sort_index())}")
    print(f"\n  ✅ DONE! Submit → {zip_path}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — RUN EVERYTHING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
text_logits,     text_f1 = train_text_model()
mm_logits,       mm_f1   = train_multimodal_model()
combined_logits          = train_combined_model()
make_final_submission(text_logits, text_f1, mm_logits, mm_f1, combined_logits)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
Device: cuda

STEP 1/4 — Text-Only RoBERTa (train→eval split)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_55/1470525225.py:476: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/tmp/ipykernel_55/1470525225.py:504: 

  Epoch 1/3  Train Loss: 0.2731
  → CLIP unfrozen for fine-tuning
  Epoch 2/3  Train Loss: 0.1837
  Epoch 3/3  Train Loss: 0.1367
  Combined model saved → /kaggle/working/best_combined.pt


/tmp/ipykernel_55/1470525225.py:528: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Combined logits saved → shape (1025, 3)

STEP 4/4 — Weighted Ensemble → Final Submission
  Model weights (F1-proportional):
    Text-only RoBERTa  : 0.334  (val F1=0.8157)
    Multimodal         : 0.333  (val F1=0.8128)
    Combined (no val)  : 0.333  (proxy F1=0.8128)

  Label dist: {0: np.int64(330), 1: np.int64(324), 2: np.int64(371)}

  ✅ DONE! Submit → /kaggle/working/submission_v2.zip
